# Summarization Transformer
### *'T5 Small for text summarization'*

Para la tarea de generación de resumenes vamos a probar el modelo `falconsai/text_summarization`. Este modelo es una variante del modelo de transformers T5, diseñado para la tarea de resumen de texto. Está adaptada y optimizada para generar resúmenes concisos y coherentes del texto de entrada. El uso principal de este modelo es generar resúmenes de texto concisos y coherentes, para aplicaciones que requieren resumir documentos extensos, artículos periodísticos y contenido textual.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "falconsai/text_summarization"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Device:", device)


c:\Users\Iñigo Peña\Desktop\Clase\2025-26\ProcesamientoDelLenguajeNatural\FinTracker\ftenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Definimos la función `summarize_texts` que procesa los textos divididos en batches de 8 y genera el resumen de cada texto. La función utiliza batching, permitiendo ajustar parámetros como la longitud del resumen (`max_new_tokens`), el beam search (`num_beams`) y penalizaciones para controlar la calidad y diversidad de los resúmenes generados.

In [5]:
from typing import List, Dict, Any

def summarize_texts(
    texts: List[str],
    batch_size: int = 8,
    max_input_tokens: int = 512,
    max_new_tokens: int = 64, # longitud del resumen generado
    min_new_tokens: int = 16,
    num_beams: int = 4,
    length_penalty: float = 1.0,
    no_repeat_ngram_size: int = 3):
    """
    Genera resúmenes para una lista de textos.
    Devuelve una lista de strings (resúmenes) en el mismo orden.
    """
    outputs = []

    clean_texts = [(t or "").strip().replace("\n", " ") for t in texts]

    for i in range(0, len(clean_texts), batch_size):
        batch = clean_texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            max_length=max_input_tokens,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                num_beams=num_beams,
                length_penalty=length_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size,
                early_stopping=True,
            )

        batch_summaries = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
        outputs.extend([s.strip() for s in batch_summaries])

    return outputs


Cargamos los datos procesados en la *E2* de los articulos financieros, y nos quedamos con el titulo y texto principal de cada uno, para despues pasar funcion que hemos hecho para resumir los textos, mostrando el texto original junto al resultado del modelo de summarization.

In [6]:
import pandas as pd


df = pd.read_csv('../../data/definitivos/INDEX_ALL_scrapped_filtrado.csv')
df = df[['article_text', 'headline']].dropna()
df['article_text'] = df['article_text'].str.lower().str.strip()
df['headline'] = df['headline'].str.lower().str.strip()
    
# Filter
df = df[df['headline'].str.split().str.len() <= 20]
df = df[df['headline'].str.split().str.len() >= 3]

df

,article_text,headline
0,the uk jobs market continues to show signs of ...,uk pay growth slows and unemployment ticks hig...
1,new data from the department from work and pen...,rising pension age hits women hardest
2,asking for workplace accommodations is often e...,how to ask for changes at work if you are neur...
3,apple has announced a major expansion of its r...,apple announces major expansion of renewables ...
4,the advent of artificial intelligence (ai) les...,"1 unstoppable stock poised to join nvidia, app..."
...,...,...
5155,social buzz: wallstreetbets stocks mostly high...,social buzz: wallstreetbets stocks mostly high...
5156,the electric vehicle (ev) plant construction m...,electric vehicles today - driving growth: new ...
5157,beijing (reuters) -chinese electric carmaker x...,"china's xpeng to recall 47,490 p7+ electric ca..."
5158,vietnam’s new vehicle market expanded slightly...,vietnam vehicle market up slightly in august


In [7]:
articulos = [t for t in df['article_text'].tolist()[:3]]

summaries = summarize_texts(articulos, batch_size=4, max_new_tokens=60)
for t, s in zip(articulos, summaries):
    print("[TEXT]")
    print(t[:300], "...")
    print("[SUMMARY]")
    print(s)
    print()


[TEXT]
the uk jobs market continues to show signs of weakness, with pay growth slowing and unemployment edging higher ahead of the autumn budget next month.
the latest data from the office for national statistics (ons), released on tuesday, showed that annual wage growth excluding bonuses in the three mon ...
[SUMMARY]
shows signs of weakness, with pay growth slowing and unemployment edging higher ahead of autumn budget next month. the figures come ahead of the government's autumn budget, which chancellor rachel reeves is due to deliver on 26 november

[TEXT]
new data from the department from work and pensions on the working patterns of people aged 50 and above showed that our working lives are getting ever longer, as the average age of leaving work rises.
this trend isn’t just down to people’s enthusiasm for work; it’s primarily caused by increases to  ...
[SUMMARY]
from work and pensions on the working patterns of people aged 50 and above. this trend isn’t just down to people’s enthu

Vamos a evaluar los resultados obtenidos mediante la métrica **ROUGE-1**, utilizada como **evaluación auxiliar cuantitativa** que complementa nuestra observación cualitativa directa de los resúmenes generados.

In [8]:
from rouge_score import rouge_scorer

def calculate_rouge_scores(predictions, references):

    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)  # Lista con ['rouge1']
    
    scores = []
    
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores.append(score['rouge1'].fmeasure)
    
    # Calcular promedios
    avg_score = sum(scores) / len(scores)
    
    return avg_score, scores

headlines = df['headline'].tolist()[:3]

avg_score, detailed_scores = calculate_rouge_scores(
    predictions=summaries,
    references=headlines
)

print("ROUGE1 Promedio:", avg_score)
print("---\n")

for i in range(len(summaries)):
    print(f"[Text {i+1}]")
    print(f"  Headline: {headlines[i]}")
    print(f"  Resumen:  {summaries[i]}")
    print(f"\n ROUGE-1: {detailed_scores[i]:.4f}")

ROUGE1 Promedio: 0.19692307692307695
---

[Text 1]
  Headline: uk pay growth slows and unemployment ticks higher ahead of budget
  Resumen:  shows signs of weakness, with pay growth slowing and unemployment edging higher ahead of autumn budget next month. the figures come ahead of the government's autumn budget, which chancellor rachel reeves is due to deliver on 26 november

 ROUGE-1: 0.3600
[Text 2]
  Headline: rising pension age hits women hardest
  Resumen:  from work and pensions on the working patterns of people aged 50 and above. this trend isn’t just down to people’s enthusiasm for work; it’s primarily caused by increases to the state pension age – a change that has particularly impacted the working lives

 ROUGE-1: 0.0769
[Text 3]
  Headline: how to ask for changes at work if you are neurodivergent
  Resumen:  for accommodations at work is often easier said than done. many people worry about being seen as “needy” or “incompetent” and as a result, continue to struggle in enviro

En la primera iteración del proyecto ([Comparativa_final.ipynb](../Primera_Resolucion/Summarization/Summarization/Comparativa_final.ipynb)), desarrollamos modelos Seq2Seq personalizados con arquitectura Encoder-Decoder con mecanismo de atención Bahdanau, evaluados sobre 100 muestras del conjunto de test. Los resultados fueron:

| Modelo | ROUGE-1 | ROUGE-2 |
|--------|---------|---------|
| FastText Frozen | 0.222905 | 0.082415 |
| FastText Fine-tuned | 0.052169 | 0.001429 |
| BERT Frozen | 0.130193 | 0.028822 |
| BERT Fine-tuned | 0 | 0 |

## Modelo Cuantizado Ollama

Queremos probar a hacer uso de un **modelo cuantizado de Ollama** para evaluar y comparar los resultados que puede dar con el resto de tecnicas y modelos que hemos empleado.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2"

def ollama_summarize(text: str) -> str:
    prompt = (
        f"Summarize the following financial news in 2 sentences. "
        f"Focus on the key facts (numbers, dates, actors)."
        f"Avoid opinions and just output the summary, without any additional commentary.\n\n"
        f"TEXT:\n{text}\n\nSUMMARY:"
    )

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"].strip()

# ejemplo
summary = ollama_summarize(articulos[0])
print(summary)


The UK jobs market saw annual wage growth slow to 4.7% in the three months to August, down from 4.8% between May and July, while the unemployment rate rose to 4.8%. The number of employees on the payroll fell by 93,000 in the year to August, but early estimates for September suggest a further decline of 100,000 on a yearly basis.


In [10]:
BASE = "http://localhost:11434"
models = requests.get(f"{BASE}/api/tags", timeout=10).json()
models.keys(), models["models"][:1]

(dict_keys(['models']),
 [{'name': 'llama3.2:latest',
   'model': 'llama3.2:latest',
   'modified_at': '2026-01-04T16:25:11.9399706+01:00',
   'size': 2019393189,
   'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '3.2B',
    'quantization_level': 'Q4_K_M'}}])